# Visualização de sequência SMPL-X em vídeo no Jupyter

Este notebook carrega uma sequência SMPL-X em `.npz`, renderiza frame a frame e gera um vídeo `.mp4` exibido inline no notebook.

## Pré-requisitos

- Arquivos do modelo SMPL-X baixados localmente
- Um arquivo `.npz` com poses
- Ambiente com suporte a renderização offscreen OpenGL/EGL/OSMesa

Formatos aceitos no `.npz`:
- SMPL-X nativo: `global_orient`, `body_pose`, `betas`, `transl`, ...
- AMASS/SMPL-X: `root_orient`, `pose_body`, `betas`, `trans`, `pose_hand`, `pose_eye`, `pose_jaw`

Arquivos com `global_rot_mats`/`local_rot_mats` ou `joint_positions_xyz` nao sao parametros SMPL-X diretos e precisam ser convertidos antes.


In [1]:
# Instale as dependências se necessário
# Em alguns ambientes também pode ser necessário: pip install PyOpenGL PyOpenGL_accelerate
%pip install -q smplx torch numpy trimesh pyrender imageio imageio-ffmpeg ipython
%pip install PyOpenGL PyOpenGL_accelerate


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import torch
import smplx
import trimesh

# Forca renderizacao offscreen via EGL antes de importar pyrender/OpenGL.
PYOPENGL_PLATFORM = os.environ.get('PYOPENGL_PLATFORM', 'egl')
os.environ['PYOPENGL_PLATFORM'] = PYOPENGL_PLATFORM

import pyrender
import imageio.v2 as imageio

print(f'PYOPENGL_PLATFORM={PYOPENGL_PLATFORM}')

from IPython.display import Video, display


PYOPENGL_PLATFORM=egl


In [3]:
DOMAIN = "10ms"
USER = "u04"
TAG = "tag18"
CAPTURE = ""

# ===== CONFIGURAÇÃO =====
# NPZ_PATH = f'output/robot_emotions_kimodo/robot_emotions_{DOMAIN}_{USER}_{TAG}{"_" + CAPTURE if CAPTURE != "" else ""}/motion_amass.npz'
NPZ_PATH = "output/robot_emotions_kimodo_generated_single/robot_emotions_10ms_u02_tag11__w000/motion_amass.npz"
MODEL_PATH = 'models_lockedhead/'
GENDER = 'neutral'   # 'neutral', 'male' ou 'female'
OUT_VIDEO = 'smplx_sequence.mp4'
FPS = 24
WIDTH = 640
HEIGHT = 480


In [4]:
# ===== CARREGAMENTO =====
data = np.load(NPZ_PATH, allow_pickle=True)
print('Chaves encontradas no arquivo:')
for k in data.files:
    print(f'  - {k}: {data[k].shape}, dtype={data[k].dtype}')

def _as_float32(name):
    return np.asarray(data[name], dtype=np.float32)


def _normalize_pose_payload():
    keys = set(data.files)

    if {'body_pose', 'global_orient'}.issubset(keys):
        pose_payload = {
            'body_pose': _as_float32('body_pose'),
            'global_orient': _as_float32('global_orient'),
        }
        for optional_name in (
            'betas',
            'transl',
            'left_hand_pose',
            'right_hand_pose',
            'jaw_pose',
            'leye_pose',
            'reye_pose',
            'expression',
        ):
            if optional_name in data:
                pose_payload[optional_name] = _as_float32(optional_name)
        return pose_payload, 'smplx_native'

    if {'pose_body', 'root_orient'}.issubset(keys):
        pose_payload = {
            'body_pose': _as_float32('pose_body'),
            'global_orient': _as_float32('root_orient'),
        }
        if 'trans' in data:
            pose_payload['transl'] = _as_float32('trans')
        if 'betas' in data:
            pose_payload['betas'] = _as_float32('betas')
        if 'pose_jaw' in data:
            pose_payload['jaw_pose'] = _as_float32('pose_jaw')
        if 'pose_eye' in data:
            pose_eye = _as_float32('pose_eye')
            if pose_eye.ndim != 2 or pose_eye.shape[1] != 6:
                raise ValueError(f'pose_eye deve ter shape (T, 6); recebido {pose_eye.shape}.')
            pose_payload['leye_pose'] = pose_eye[:, :3]
            pose_payload['reye_pose'] = pose_eye[:, 3:]
        if 'pose_hand' in data:
            pose_hand = _as_float32('pose_hand')
            if pose_hand.ndim != 2 or pose_hand.shape[1] != 90:
                raise ValueError(f'pose_hand deve ter shape (T, 90); recebido {pose_hand.shape}.')
            pose_payload['left_hand_pose'] = pose_hand[:, :45]
            pose_payload['right_hand_pose'] = pose_hand[:, 45:]
        return pose_payload, 'amass'

    if {'global_rot_mats', 'posed_joints'}.issubset(keys) or {'local_rot_mats', 'root_positions'}.issubset(keys):
        raise KeyError(
            'Este arquivo parece ser um motion.npz bruto do Kimodo. Para este notebook, use o export AMASS/SMPL-X correspondente, por exemplo `kimodo/output/kimodo/_amass.npz`.'
        )

    if {'joint_positions_xyz', 'joint_names_3d'}.issubset(keys):
        raise KeyError(
            'Este arquivo parece ser um pose3d.npz do pipeline (`PoseSequence3D`). Ele contem joints 3D, nao parametros SMPL-X. Converta antes para um NPZ SMPL-X/AMASS.'
        )

    raise KeyError(
        'O arquivo .npz precisa conter `body_pose` + `global_orient` ou `pose_body` + `root_orient`.'
    )


pose_data, pose_format = _normalize_pose_payload()
body_pose = pose_data['body_pose']
global_orient = pose_data['global_orient']

if body_pose.ndim != 2 or body_pose.shape[1] != 63:
    raise ValueError(f'body_pose deve ter shape (T, 63); recebido {body_pose.shape}.')
if global_orient.shape != (body_pose.shape[0], 3):
    raise ValueError(
        f'global_orient deve ter shape {(body_pose.shape[0], 3)}; recebido {global_orient.shape}.'
    )

T = len(body_pose)
print(f'\nFormato detectado: {pose_format}')
print('Campos normalizados para o modelo SMPL-X:')
for k, v in pose_data.items():
    print(f'  - {k}: {v.shape}, dtype={v.dtype}')
print(f'\nNúmero de frames: {T}')


Chaves encontradas no arquivo:
  - trans: (150, 3), dtype=float32
  - root_orient: (150, 3), dtype=float32
  - pose_body: (150, 63), dtype=float32
  - gender: (), dtype=<U7
  - surface_model_type: (), dtype=<U5
  - betas: (16,), dtype=float32
  - num_betas: (), dtype=int64
  - mocap_frame_rate: (), dtype=float64
  - pose_jaw: (150, 3), dtype=float64
  - pose_eye: (150, 6), dtype=float64
  - pose_hand: (150, 90), dtype=float64
  - mocap_time_length: (), dtype=float64

Formato detectado: amass
Campos normalizados para o modelo SMPL-X:
  - body_pose: (150, 63), dtype=float32
  - global_orient: (150, 3), dtype=float32
  - transl: (150, 3), dtype=float32
  - betas: (16,), dtype=float32
  - jaw_pose: (150, 3), dtype=float32
  - leye_pose: (150, 3), dtype=float32
  - reye_pose: (150, 3), dtype=float32
  - left_hand_pose: (150, 45), dtype=float32
  - right_hand_pose: (150, 45), dtype=float32

Número de frames: 150


In [5]:
# ===== UTILITÁRIOS =====
def get_optional_tensor(name, t, total_frames):
    if name not in pose_data:
        return None

    arr = pose_data[name]

    if arr.ndim == 1:
        arr = arr[None]
    elif arr.shape[0] == total_frames:
        arr = arr[t:t+1]
    else:
        arr = arr[:1]

    return torch.tensor(arr, dtype=torch.float32)


def _fit_feature_dim(arr, target_dim):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[None]
    current_dim = int(arr.shape[-1])
    if current_dim == target_dim:
        return arr
    if current_dim > target_dim:
        return arr[..., :target_dim]
    pad_width = [(0, 0)] * arr.ndim
    pad_width[-1] = (0, target_dim - current_dim)
    return np.pad(arr, pad_width, mode='constant')


def get_betas(t, total_frames):
    target_dim = int(getattr(model, 'num_betas', 10))
    if 'betas' not in pose_data:
        arr = np.zeros((1, target_dim), dtype=np.float32)
    else:
        arr = _fit_feature_dim(pose_data['betas'], target_dim)
        if arr.ndim == 1:
            arr = arr[None]
        elif arr.shape[0] == total_frames:
            arr = arr[t:t+1]
        else:
            arr = arr[:1]

    return torch.tensor(arr, dtype=torch.float32)


def get_expression(t, total_frames):
    target_dim = int(getattr(model, 'num_expression_coeffs', 10))
    if 'expression' not in pose_data:
        arr = np.zeros((1, target_dim), dtype=np.float32)
    else:
        arr = _fit_feature_dim(pose_data['expression'], target_dim)
        if arr.ndim == 1:
            arr = arr[None]
        elif arr.shape[0] == total_frames:
            arr = arr[t:t+1]
        else:
            arr = arr[:1]

    return torch.tensor(arr, dtype=torch.float32)


def create_offscreen_renderer(width, height):
    try:
        return pyrender.OffscreenRenderer(viewport_width=width, viewport_height=height)
    except Exception as exc:
        raise RuntimeError(
            'Falha ao criar o renderer offscreen. Neste ambiente, use `PYOPENGL_PLATFORM=egl`. '
            'Se voce acabou de alterar essa configuracao no notebook, reinicie o kernel e execute tudo novamente desde a primeira celula.'
        ) from exc


In [6]:
# ===== MODELO =====
model = smplx.create(
    model_path=MODEL_PATH,
    model_type='smplx',
    gender=GENDER,
    use_pca=False,
    batch_size=1
)

print('Modelo carregado com sucesso.')
print(f'num_betas suportado pelo modelo: {model.num_betas}')
print(f'num_expression_coeffs suportado pelo modelo: {model.num_expression_coeffs}')
print(f'Número de vértices: {model.get_num_verts()}')
print(f'Número de faces: {len(model.faces)}')


Modelo carregado com sucesso.
num_betas suportado pelo modelo: 10
num_expression_coeffs suportado pelo modelo: 10
Número de vértices: 10475
Número de faces: 20908


In [7]:
# ===== CENA E RENDERER =====

# Ajuste da câmera
CAM_X = 0.0
CAM_Y = 2.0
CAM_Z = 1.5
ROT_X = np.pi / 3
ROT_Y = 0.0
ROT_Z = np.pi
BG_COLOR = [1.0, 1.0, 1.0, 1.0]
AMBIENT_LIGHT = [0.35, 0.35, 0.35]
DIRECTIONAL_LIGHT_INTENSITY = 2.0

scene = pyrender.Scene(bg_color=BG_COLOR, ambient_light=AMBIENT_LIGHT)

camera = pyrender.PerspectiveCamera(yfov=np.pi / 1.7)
cam_pose = np.array([
    [1, 0, 0, CAM_X],
    [0, 1, 0, CAM_Y],
    [0, 0, 1, CAM_Z],
    [0, 0, 0, 1],
], dtype=np.float32)

rotation_x = np.array([
    [1, 0, 0, 0],
    [0, np.cos(ROT_X), -np.sin(ROT_X), 0],
    [0, np.sin(ROT_X),  np.cos(ROT_X), 0],
    [0, 0, 0, 1],
], dtype=np.float32)

rotation_y = np.array([
    [ np.cos(ROT_Y), 0, np.sin(ROT_Y), 0],
    [0, 1, 0, 0],
    [-np.sin(ROT_Y), 0, np.cos(ROT_Y), 0],
    [0, 0, 0, 1],
], dtype=np.float32)

rotation_z = np.array([
    [np.cos(ROT_Z), -np.sin(ROT_Z), 0, 0],
    [np.sin(ROT_Z),  np.cos(ROT_Z), 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
], dtype=np.float32)

final_cam = cam_pose @ rotation_z @ rotation_y @ rotation_x
scene.add(camera, pose=final_cam)

light = pyrender.DirectionalLight(color=np.ones(3), intensity=DIRECTIONAL_LIGHT_INTENSITY)
scene.add(light, pose=cam_pose)

print(f'Criando renderer com backend {PYOPENGL_PLATFORM!r}...')
renderer = create_offscreen_renderer(WIDTH, HEIGHT)
writer = imageio.get_writer(OUT_VIDEO, fps=FPS)

mesh_node = None


Criando renderer com backend 'egl'...


In [8]:
# ===== LOOP DE RENDERIZAÇÃO =====
for t in range(T):
    body_pose = torch.tensor(pose_data['body_pose'][t:t+1], dtype=torch.float32)
    global_orient = torch.tensor(pose_data['global_orient'][t:t+1], dtype=torch.float32)

    output = model(
        betas=get_betas(t, T),
        body_pose=body_pose,
        global_orient=global_orient,
        transl=get_optional_tensor('transl', t, T),
        left_hand_pose=get_optional_tensor('left_hand_pose', t, T),
        right_hand_pose=get_optional_tensor('right_hand_pose', t, T),
        jaw_pose=get_optional_tensor('jaw_pose', t, T),
        leye_pose=get_optional_tensor('leye_pose', t, T),
        reye_pose=get_optional_tensor('reye_pose', t, T),
        expression=get_expression(t, T),
        return_verts=True
    )

    verts = output.vertices.detach().cpu().numpy().squeeze()
    tri_mesh = trimesh.Trimesh(vertices=verts, faces=model.faces, process=False)
    render_mesh = pyrender.Mesh.from_trimesh(tri_mesh, smooth=False)

    if mesh_node is not None:
        scene.remove_node(mesh_node)
    mesh_node = scene.add(render_mesh)

    color, _ = renderer.render(scene)
    writer.append_data(color)

    if (t + 1) % 10 == 0 or t == T - 1:
        print(f'Renderizado: {t + 1}/{T}')

writer.close()
renderer.delete()
print(f'Vídeo salvo em: {OUT_VIDEO}')


Renderizado: 10/150
Renderizado: 20/150
Renderizado: 30/150
Renderizado: 40/150
Renderizado: 50/150
Renderizado: 60/150
Renderizado: 70/150
Renderizado: 80/150
Renderizado: 90/150
Renderizado: 100/150
Renderizado: 110/150
Renderizado: 120/150
Renderizado: 130/150
Renderizado: 140/150
Renderizado: 150/150
Vídeo salvo em: smplx_sequence.mp4


In [9]:
# ===== EXIBIÇÃO INLINE =====
display(Video(OUT_VIDEO, embed=True))


## Notas de depuração

Se `pyrender.OffscreenRenderer(...)` falhar, o problema normalmente é o backend OpenGL offscreen do ambiente. Alternativas:

- usar `PYOPENGL_PLATFORM=egl` antes do import de `pyrender` e reiniciar o kernel
- usar um ambiente com EGL ou OSMesa
- testar no Colab
- exportar `.obj` por frame em vez de renderizar vídeo

Se a malha aparecer deformada, verifique:

- se `body_pose` está em axis-angle e tem shape `(T, 63)`
- se `global_orient` tem shape `(T, 3)`
- se o arquivo estiver em AMASS, use `pose_body`/`root_orient` ou o `_amass.npz` gerado pelo Kimodo
- se `betas` é `(10,)` ou `(T, 10)`
- se as poses de mão não estão em formato PCA quando `use_pca=False`
- se voce nao apontou o notebook para `motion.npz` bruto (`global_rot_mats`, `local_rot_mats`, ...) em vez do export SMPL-X
